# Model Training and Cross-Validation

Three supervised learning methods are trained in this notebook. They are Logistic Regression, Decision Tree and Random forest. They will train using the preprocessed data from milestone 1

Five-fold stratified cross-validation is used to check how consistently each model performs.

## 1. Import libraries


In [6]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

## 2. Load the preprocessed data


In [7]:
milestone1_data = joblib.load("data/processed/prepared_data.joblib")

X_train = milestone1_data["X_train"]
X_test = milestone1_data["X_test"]
y_train = milestone1_data["y_train"]
y_test = milestone1_data["y_test"]

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Training features:", X_train.shape[1])

Training rows: 7818
Testing rows: 1955
Training features: 31


## 3. Create the 3 models


In [8]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=202300408),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=202300408, n_jobs=1)
}

pipelines = {}

print("Models are ready.")

Models are ready.


## 4. Apply five-fold cross-validation

Model is trained on 5 different parts of data, F1 used as evaluation metric because classes are imbalanced.

In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=202300408)

cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1", n_jobs=1)
    cv_results[name] = scores

    print(name)
    print("F1 scores:", scores)
    print("Average F1:", scores.mean())
    print("Standard deviation:", scores.std(), "\n")

Logistic Regression
F1 scores: [0.04743083 0.05363985 0.05323194 0.07633588 0.0620155 ]
Average F1: 0.05853079953697664
Standard deviation: 0.01004363123420605 

Decision Tree
F1 scores: [0.3254902  0.28629032 0.2800789  0.33904762 0.29296875]
Average F1: 0.30477515663404126
Standard deviation: 0.023215082091027434 

Random Forest
F1 scores: [0.19863014 0.18120805 0.15540541 0.23762376 0.20408163]
Average F1: 0.19538979822245614
Standard deviation: 0.027091065154815634 



## 5. Train models on the full training set


In [10]:
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(name," has been trained.")

milestone2_data = {
    "X_test": X_test,
    "y_test": y_test,
    "models": trained_models,
    "cv_results": cv_results
}

joblib.dump(milestone2_data, "data/processed/trained_models.joblib")

Logistic Regression  has been trained.
Decision Tree  has been trained.
Random Forest  has been trained.


['data/processed/trained_models.joblib']

## 6. Cross-validation summary


In [11]:
cv_summary = pd.DataFrame({
    "Model": list(cv_results.keys()),
    "Average CV F1-score": [scores.mean() for scores in cv_results.values()],
    "CV F1 standard deviation": [scores.std() for scores in cv_results.values()]
})

cv_summary = cv_summary.sort_values(by="Average CV F1-score", ascending=False)
print(cv_summary.to_string(index=False))

              Model  Average CV F1-score  CV F1 standard deviation
      Decision Tree             0.304775                  0.023215
      Random Forest             0.195390                  0.027091
Logistic Regression             0.058531                  0.010044
